<a href="https://colab.research.google.com/github/gopika-vit/Projects-AI/blob/projects-in-colab/Bidirectional_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Download the zip from GitHub (FSDD source)
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git

# 2. Set your path to where the recordings are
my_data_path = "/content/free-spoken-digit-dataset/recordings"

# 3. Check if it worked
import os
files = os.listdir(my_data_path)
print(f"Success! Found {len(files)} files in {my_data_path}")

Cloning into 'free-spoken-digit-dataset'...
remote: Enumerating objects: 4260, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 4260 (delta 25), reused 8 (delta 8), pack-reused 4212 (from 1)
Receiving objects: 100% (4260/4260), 30.38 MiB | 18.34 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Success! Found 3000 files in /content/free-spoken-digit-dataset/recordings


In [ ]:
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Define the Dataset
class SpokenDigitDataset(Dataset):
    def __init__(self, data_path, max_len=50):
        self.data_path = data_path
        self.file_list = [f for f in os.listdir(data_path) if f.endswith('.wav')]
        self.max_len = max_len

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = os.path.join(self.data_path, self.file_list[idx])
        # Load audio at 8kHz
        audio, sr = librosa.load(file_path, sr=8000)
        # Extract 13 MFCC features
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

        # Padding/Truncating to keep lengths consistent
        if mfcc.shape[1] < self.max_len:
            pad_width = self.max_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :self.max_len]

        return torch.FloatTensor(mfcc.T), torch.tensor(int(self.file_list[idx].split('_')[0]))

In [ ]:
# Define the BiRNN Model
class AutismAudioBiRNN(nn.Module):
    def __init__(self, input_size=13, hidden_size=64, num_layers=2, num_classes=10):
        super(AutismAudioBiRNN, self).__init__()
        # Bidirectional LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True, dropout=0.3)
        # Multiply hidden_size by 2 for Bidirectional concatenation
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Classify based on the final time step
        out = self.fc(lstm_out[:, -1, :])
        return out

In [ ]:
# Use the path from the git clone
my_data_path = "/content/free-spoken-digit-dataset/recordings"

dataset = SpokenDigitDataset(data_path=my_data_path)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Detect if GPU is available (Go to Runtime -> Change runtime type -> T4 GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutismAudioBiRNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Setup complete. Ready to train on {device}!")

Setup complete. Ready to train on cpu!


In [ ]:
num_epochs = 15 # We'll do 15 rounds

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward Pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward Pass (Optimization)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Training finished!")

Starting training...


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2012
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1873
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1896
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1936
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1988
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1987
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py

Epoch [1/15], Loss: 2.3052
Epoch [2/15], Loss: 2.3043
Epoch [3/15], Loss: 2.3038
Epoch [4/15], Loss: 2.3041
Epoch [5/15], Loss: 2.3039
Epoch [6/15], Loss: 2.3039
Epoch [7/15], Loss: 2.3036
Epoch [8/15], Loss: 2.3036
Epoch [9/15], Loss: 2.3034
Epoch [10/15], Loss: 2.3033
Epoch [11/15], Loss: 2.3031
Epoch [12/15], Loss: 2.3027
Epoch [13/15], Loss: 2.1637
Epoch [14/15], Loss: 1.6282
Epoch [15/15], Loss: 1.3009
Training finished!


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the BiRNN on the dataset: {100 * correct / total:.2f}%')

Accuracy of the BiRNN on the dataset: 41.40%
